In [23]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
with h5py.File(filename, 'r') as f:
    logOE_matrix = f['data/logOE_matrix'][:]
    gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
gene_names = [x.decode('UTF-8') for x in gene_names]


category_list  = {
    "Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"
}
category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()

In [24]:
# Get category-sample indexes
all_cat_idxs = []
for element in category_list:
    indexes = []
    for i, name in enumerate(category_names):
        if element.lower() in name.lower():
            indexes.append(i)
    all_cat_idxs.append(indexes)

# Initialize the dictionaries to store the averages
gene_tier1cat_avgs = []

for gene_row in logOE_matrix:
    values = [[gene_row[i] for i in sub_list] for sub_list in all_cat_idxs]
    means = []
    for value in values:
        if len(value) == 0:
            means.append(np.nan)
        else:
            mean = sum(value)/len(value)
            means.append(mean)
    gene_tier1cat_avgs.append(means)


In [25]:
len(gene_tier1cat_avgs)

57447

In [26]:
len(gene_tier1cat_avgs[0])

127

In [27]:
from IPython.display import display

df = pd.DataFrame(gene_tier1cat_avgs, index=gene_names)
df.columns = category_list
display(df)

,Pancreatic Islet,Macrophage,Renal Cortex,SKNSH,Medula,MG63,Melanocyte,Motor Neuron,Retina,Liver,...,Breast,LNCAP,Airway Smooth Muscle,Bone Marrow,Testis,NHBE,293F,Skeletal Muscle,HUVEC,LHCNM2
A1BG,0.904145,0.958343,0.126199,0.896957,NaN,0.379963,0.903993,0.878593,0.399846,2.668668,...,0.509562,0.633759,0.657144,1.064748,0.165445,0.228951,0.423089,0.269254,0.312708,0.388097
A1BG-AS1,1.114086,1.530616,0.155662,1.162818,NaN,0.880483,1.674517,0.999136,0.633474,0.703795,...,0.906342,1.316072,1.439635,0.768679,0.195265,0.624976,1.027124,0.316963,0.696092,1.179094
A1CF,2.086442,0.060457,1.489218,0.071461,NaN,0.053629,0.070890,0.069592,0.059816,3.209776,...,0.079024,0.062288,0.039662,0.154721,0.022998,0.044014,0.053999,0.080976,0.045534,0.064632
A2M,0.546406,1.678592,2.061387,0.009902,NaN,0.008920,1.135664,0.277894,0.809409,1.997129,...,0.588174,0.009428,1.474954,0.170736,1.030224,0.081944,0.012005,1.276533,0.217915,0.030852
A2M-AS1,0.930792,1.285306,0.315589,0.141459,NaN,0.066793,0.664648,0.851478,0.796921,1.443540,...,0.502568,1.008432,0.811750,0.550690,0.494885,0.201894,0.327507,1.500945,0.628132,0.851293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11B,0.997061,0.867326,1.051246,0.954070,NaN,0.795737,0.829231,1.155680,1.087419,1.025150,...,0.747509,0.901631,1.126946,0.807482,1.095335,0.780870,1.123242,2.358598,0.958861,0.944999
ZYX,0.374880,1.380199,0.772848,1.392962,NaN,1.187411,0.889033,0.867725,0.618271,0.824058,...,0.911611,0.552858,1.576343,0.805454,0.729608,1.285870,0.593512,0.583755,1.452258,1.309560
ZYXP1,0.941739,0.773403,0.158562,0.680553,NaN,0.448242,0.577100,0.495699,0.453785,0.660689,...,0.708718,0.456105,0.517446,1.085080,0.195146,0.681558,0.298976,0.573350,0.714047,0.862428
ZZEF1,0.870000,1.234872,1.503008,0.714966,NaN,0.891057,1.186310,0.856583,1.065050,0.881304,...,0.910215,0.972171,1.001760,0.971259,0.879547,0.793562,0.978547,1.145550,1.085488,0.929565


In [28]:
# loop through each column and get the top 10 highest values and their indexes
for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Pancreatic Islet":
INS        6.517171
GCG        6.380697
SLC30A8    6.187919
G6PC2      6.097876
PNLIP      5.982218
CELA3A     5.912454
KCNK16     5.862052
PPY        5.831182
CPA1       5.803547
PRSS1      5.755286
Name: Pancreatic Islet, dtype: float64
Indexes for Column "Pancreatic Islet":
Index(['INS', 'GCG', 'SLC30A8', 'G6PC2', 'PNLIP', 'CELA3A', 'KCNK16', 'PPY',
       'CPA1', 'PRSS1'],
      dtype='object')

Top 10 values in Column "Macrophage":
MMP9        3.524722
ADAMDEC1    3.430307
GPR84       3.401923
PLA2G7      3.388287
SLAMF8      3.332979
LILRB4      3.307894
CLEC5A      3.295825
CD300C      3.244722
SIGLEC9     3.232163
CCR1        3.224746
Name: Macrophage, dtype: float64
Indexes for Column "Macrophage":
Index(['MMP9', 'ADAMDEC1', 'GPR84', 'PLA2G7', 'SLAMF8', 'LILRB4', 'CLEC5A',
       'CD300C', 'SIGLEC9', 'CCR1'],
      dtype='object')

Top 10 values in Column "Renal Cortex":
UMOD               7.171689
SLC22A12           6.958868
SLC22A6

Top 10 values in Column "Chondrocyte":
ACAN      5.882402
CCN6      5.616623
HAPLN1    5.395976
ECRG4     4.887687
COL2A1    4.642653
COMP      4.622158
MMP3      4.412000
PENK      4.400435
ITGA10    4.379641
ITIH6     4.356096
Name: Chondrocyte, dtype: float64
Indexes for Column "Chondrocyte":
Index(['ACAN', 'CCN6', 'HAPLN1', 'ECRG4', 'COL2A1', 'COMP', 'MMP3', 'PENK',
       'ITGA10', 'ITIH6'],
      dtype='object')

Top 10 values in Column "HT29":
UGT1A10    4.879220
UGT1A8     4.793386
NMUR2      4.529324
KRT20      4.316394
TRIM31     4.114050
MYRFL      4.079474
HOTTIP     3.881980
KLK6       3.823488
NTSR1      3.806378
VSIG2      3.715420
Name: HT29, dtype: float64
Indexes for Column "HT29":
Index(['UGT1A10', 'UGT1A8', 'NMUR2', 'KRT20', 'TRIM31', 'MYRFL', 'HOTTIP',
       'KLK6', 'NTSR1', 'VSIG2'],
      dtype='object')

Top 10 values in Column "Esophagus":
KRT13              4.897480
SPRR3              4.418927
KRT4               4.248369
CRNN               3.861023
IL36A     

Top 10 values in Column "HACAT":
KRT13              5.576353
SERPINB13          5.012746
ENSG00000225867    4.914731
WNT10A             4.544580
ENSG00000223829    4.400080
KRT4               4.264268
HOXC12             3.916244
ENSG00000271392    3.890521
ENSG00000273760    3.854859
UGT1A6             3.798411
Name: HACAT, dtype: float64
Indexes for Column "HACAT":
Index(['KRT13', 'SERPINB13', 'ENSG00000225867', 'WNT10A', 'ENSG00000223829',
       'KRT4', 'HOXC12', 'ENSG00000271392', 'ENSG00000273760', 'UGT1A6'],
      dtype='object')

Top 10 values in Column "HEPG2":
ENSG00000230952    5.047049
BPIFB2             4.844706
AFP                4.773207
ENSG00000254054    4.768602
NR5A1              4.664911
ENSG00000260573    4.474278
NOTUM              4.332697
SP5                4.321472
DGKK               4.234709
SSUH2              4.127343
Name: HEPG2, dtype: float64
Indexes for Column "HEPG2":
Index(['ENSG00000230952', 'BPIFB2', 'AFP', 'ENSG00000254054', 'NR5A1',
       'ENSG00000

Top 10 values in Column "NHBE":
TMPRSS11D          4.771653
PTHLH              4.525721
CXCL17             4.517544
BPIFA1             4.491991
UPK1B              4.429559
SPRR3              4.297878
KRT13              4.206479
ATP12A             4.196498
ENSG00000253339    4.191707
BPIFA2             4.021317
Name: NHBE, dtype: float64
Indexes for Column "NHBE":
Index(['TMPRSS11D', 'PTHLH', 'CXCL17', 'BPIFA1', 'UPK1B', 'SPRR3', 'KRT13',
       'ATP12A', 'ENSG00000253339', 'BPIFA2'],
      dtype='object')

Top 10 values in Column "293F":
IRS4         4.366126
IRS4-AS1     3.840856
HOXD13       3.810112
HOXD10       3.397886
HOXD11       3.397428
CALCB        3.317992
FOXCUT       3.302458
SLIT2-IT1    3.271360
RADX         3.248082
HMX1         3.107774
Name: 293F, dtype: float64
Indexes for Column "293F":
Index(['IRS4', 'IRS4-AS1', 'HOXD13', 'HOXD10', 'HOXD11', 'CALCB', 'FOXCUT',
       'SLIT2-IT1', 'RADX', 'HMX1'],
      dtype='object')

Top 10 values in Column "Skeletal Muscle":
MYH

---

In [37]:
# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()
# print(category_dict)

gene_system_idx = {}
for category, keywords in category_dict.items():
    category_indexes = []
    for i, sample_name in enumerate(category_names):
        for keyword in keywords:
            if keyword.lower() in sample_name.lower():
                category_indexes.append(i)
                break
    gene_system_idx[category] = category_indexes
    
gene_cat_avgs = []
for gene_row in logOE_matrix:
    cat_avg = []

    for system, indexes in gene_system_idx.items():
        values = [gene_row[index] for index in indexes]
        mean = sum(values)/len(values)
        cat_avg.append(mean)
    gene_cat_avgs.append(cat_avg)
    
print(len(gene_cat_avgs))

57447


In [38]:
len(gene_cat_avgs[1])

9

In [39]:
from IPython.display import display
system_list = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]

df2 = pd.DataFrame(gene_cat_avgs, index=gene_names)
df2.columns = system_list
display(df2)

,Cardiovascular System,Connective Tissue,Digestive System,Immune System,Integumentary System,Muscular System,Nervous System,Respiratory System,Urogenital System
A1BG,0.305966,0.708426,1.248448,0.844066,0.640211,0.561007,0.546976,0.528425,0.503570
A1BG-AS1,0.558774,0.756588,0.537718,1.305408,1.010191,0.933012,0.777112,0.842564,0.932369
A1CF,0.045954,0.100313,1.982509,0.059649,0.068101,0.064614,0.054311,0.174963,0.141632
A2M,1.107140,0.493541,1.325945,0.972785,0.528030,0.857117,0.778685,0.546899,0.273653
A2M-AS1,1.317304,0.652329,0.977607,0.832580,0.435369,0.994019,0.732597,0.696619,0.524775
...,...,...,...,...,...,...,...,...,...
ZYG11B,1.238867,0.934100,0.870405,0.858955,0.933131,1.711243,1.271286,0.784692,0.822411
ZYX,1.093685,0.881272,0.861870,1.192684,1.124079,0.925389,0.747380,1.042486,0.818319
ZYXP1,0.545229,0.757613,0.596743,0.695417,0.611088,0.549441,0.408711,0.596826,0.694336
ZZEF1,1.049889,0.898072,0.922819,1.183043,0.920057,1.004202,0.939314,0.919760,0.896280


In [41]:
# loop through each column and get the top 10 highest values and their indexes
for col in df2.columns:
    top_10 = df2[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Cardiovascular System":
CLEC14A     3.260261
ANKRD1      3.248686
CDH5        3.216833
ECSCR       3.185909
VWF         3.139767
MYCT1       3.089548
ARHGEF15    3.089133
BMX         3.039261
MMRN2       2.997747
SCN5A       2.984509
Name: Cardiovascular System, dtype: float64
Indexes for Column "Cardiovascular System":
Index(['CLEC14A', 'ANKRD1', 'CDH5', 'ECSCR', 'VWF', 'MYCT1', 'ARHGEF15', 'BMX',
       'MMRN2', 'SCN5A'],
      dtype='object')

Top 10 values in Column "Connective Tissue":
HEMGN     1.842342
NFE2      1.839974
SPTA1     1.810053
GYPE      1.785223
STAT5A    1.781637
GFI1B     1.771943
RHAG      1.766371
GYPA      1.765118
GATA1     1.758689
GYPC      1.756836
Name: Connective Tissue, dtype: float64
Indexes for Column "Connective Tissue":
Index(['HEMGN', 'NFE2', 'SPTA1', 'GYPE', 'STAT5A', 'GFI1B', 'RHAG', 'GYPA',
       'GATA1', 'GYPC'],
      dtype='object')

Top 10 values in Column "Digestive System":
HNF4A     2.146045
FOXA3     2.128221
TRI